# MyMusic AI — temporary free GPU model server

This notebook is for **experiments only**. It uses a temporary Google Colab GPU session to run ACE-Step 1.5. Free GPU access is not guaranteed; sessions can disconnect and model downloads may take time. Do not use this as a production server.

**Security:** The tunnel URL is public. This notebook requires an API key, but use only temporary test data. Never paste the API key into GitHub or frontend code. Stop the tunnel and runtime when finished.

Official project: https://github.com/ace-step/ACE-Step-1.5


In [ ]:
# Check that Colab assigned a GPU. In Colab, select Runtime > Change runtime type > GPU first.
import torch, shutil, os, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
else: print('No GPU detected. Stop here and select a GPU runtime if one is available.')


In [ ]:
# Install ACE-Step 1.5 from its official repository.
%cd /content
!git clone https://github.com/ace-step/ACE-Step-1.5.git
%cd /content/ACE-Step-1.5
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ.get('PATH','')
!uv sync


## Start the authenticated model API

Enter a fresh temporary API key below. Keep it private. The same value must be set as `ACE_STEP_API_KEY` in your MyMusic AI backend environment. The backend must be able to reach the public tunnel URL printed in the next step.

In [ ]:
import getpass, os, subprocess, time, urllib.request
api_key = getpass.getpass('Temporary ACE-Step API key (do not share): ').strip()
if len(api_key) < 24:
    raise ValueError('Use a long random temporary key (at least 24 characters).')
os.environ['ACESTEP_API_KEY'] = api_key
os.environ['ACESTEP_API_HOST'] = '127.0.0.1'
os.environ['ACESTEP_API_PORT'] = '8001'
log = open('/content/acestep-api.log', 'w')
api_process = subprocess.Popen(['uv','run','acestep-api'], stdout=log, stderr=subprocess.STDOUT, cwd='/content/ACE-Step-1.5', env=os.environ.copy())
print('API process started. Waiting for initialization; first startup downloads model files.')
ready = False
for _ in range(180):
    if api_process.poll() is not None: break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8001/health', timeout=3) as r:
            if r.status == 200: ready = True; break
    except Exception: pass
    time.sleep(5)
print('API health endpoint responded:', ready)
if not ready:
    print('The model may still be loading or startup failed. Inspect /content/acestep-api.log before continuing.')


In [ ]:
# Install Cloudflare Tunnel client and create a temporary HTTPS endpoint.
# Only do this after the authenticated API server is running.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import re, threading
tunnel_log = open('/content/cloudflared.log', 'w')
tunnel_process = subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:8001','--no-autoupdate'], stdout=tunnel_log, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(60):
    time.sleep(2)
    try:
        txt = open('/content/cloudflared.log').read()
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
        if match: public_url = match.group(0); break
    except Exception: pass
print('Temporary model API URL:', public_url or 'Not detected; inspect /content/cloudflared.log')
print('Backend environment variables to configure securely:')
print('MUSIC_MODEL_BACKEND=ace_step')
print('ACE_STEP_BASE_URL=' + (public_url or 'https://YOUR-TUNNEL.trycloudflare.com'))
print('ACE_STEP_API_KEY=<same temporary key entered above; set as a secret, do not paste here>')
print('ACE_STEP_TIMEOUT_SECONDS=900')
print('ACE_STEP_POLL_SECONDS=2')


## Stop the session

When finished, stop `tunnel_process` and `api_process` (or disconnect/delete the Colab runtime). The tunnel URL is temporary and will change after restart. A free session may stop at any time. Do not expose this setup to public users or remove MyMusic AI's development-only generation guard to work around missing production authentication.

In [ ]:
for proc in (globals().get('tunnel_process'), globals().get('api_process')):
    if proc is not None and proc.poll() is None: proc.terminate()
print('Stop requested for API and tunnel processes. Disconnect the runtime when done.')
